# LINALDB embedded: real handwritten-digit classification

This notebook is the Jupyter counterpart of
[`digit_classification_embedded.py`](digit_classification_embedded.py) —
the embedded-mode version of
[`clients/python/examples/digit_classification.py`](../../python/examples/digit_classification.py).

It replays the real UCI handwritten-digits classification workflow from
[`examples/hdf5_digit_classification.lnl`](../../../examples/hdf5_digit_classification.lnl)
through an **in-process** `linaldb_embedded.Db()` — no `linal serve`
subprocess, no HTTP — then independently recomputes the classification in
plain numpy from the same dataset (read directly off disk), and confirms
the two paths agree exactly.

This uses real data: 8x8 grayscale handwritten-digit bitmaps from the UCI
"Optical Recognition of Handwritten Digits" dataset (Alpaydin & Kaynak,
1998), not synthetic samples.

## 1. Locate the repo and import the embedded engine

In [1]:
import os
import sys
from pathlib import Path

import numpy as np

# Walk upward from the current working directory (nbconvert's default cwd
# for a notebook is the notebook's own directory, but this stays correct
# even if run from elsewhere) until we find the repo root -- identified by
# the real .lnl showcase script this notebook replays.
_candidates = [Path.cwd()] + list(Path.cwd().parents)
REPO_ROOT = next(p for p in _candidates if (p / "examples" / "hdf5_digit_classification.lnl").exists())
LNL_SCRIPT = REPO_ROOT / "examples" / "hdf5_digit_classification.lnl"
DATABASE = "hdf5_digit_classification"

sys.path.insert(0, str(REPO_ROOT / "clients" / "python-embedded" / "python"))
import linaldb_embedded as linaldb  # noqa: E402

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"LNL_SCRIPT = {LNL_SCRIPT} (exists: {LNL_SCRIPT.exists()})")

REPO_ROOT = /Users/nicolasbalaguera/dev/linaldb/linal-db-rs
LNL_SCRIPT = /Users/nicolasbalaguera/dev/linaldb/linal-db-rs/examples/hdf5_digit_classification.lnl (exists: True)


## 2. Open an embedded instance

No server process, no port, no network — `Db()` links the engine
directly into this Python process. Persistence (`SAVE DATASET` below)
writes to `./data` relative to the current working directory, exactly
like the CLI, so we `chdir` to the repo root first (matching how the real
`.lnl` file's relative HDF5 path is meant to resolve, `examples/data/...`).

In [2]:
os.chdir(REPO_ROOT)
db = linaldb.Db()
print("Embedded LINALDB instance ready. active_db =", db.active_db())

Embedded LINALDB instance ready. active_db = default


## 3. Replay the real DSL workflow in-process

Executes each statement of the real `.lnl` file directly against `db` —
the same statements `linal run examples/hdf5_digit_classification.lnl`
would run from the CLI. Mirrors `linal run`'s own multi-line statement
joiner (`src/main.rs`): accumulate lines, track paren balance, execute
once balance returns to zero.

In [3]:
def replay_lnl_file(db, path):
    current = ""
    paren_balance = 0
    n_statements = 0
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not current:
            if not line or line.startswith("--"):
                continue
        current = f"{current} {line}".strip() if current else line
        paren_balance += line.count("(") - line.count(")")
        if paren_balance == 0:
            db.execute(current)
            n_statements += 1
            current = ""
    return n_statements


n = replay_lnl_file(db, LNL_SCRIPT)
print(f"Replayed {n} real DSL statements in-process.")

Replayed 64 real DSL statements in-process.


## 4. Run the real classification query

Nearest-centroid classification by cosine similarity: for each held-out
query digit, find the reference centroid it's most similar to (an
index-accelerated similarity join, deduplicated to the top-1 match per
query via a window function — the same query
`examples/hdf5_digit_classification.lnl` itself runs).

The `.lnl` file's own last line does `USE default`, so we switch back to
`hdf5_digit_classification` first — the embedded-mode equivalent of the
HTTP example's per-request `X-Linal-Database` header scoping.

In [4]:
db.execute(f"USE {DATABASE}")

classify_sql = (
    "WITH classified AS ("
    "SELECT query_digits.digit_id AS digit_id, query_digits.true_label AS true_label, "
    "reference_centroids.digit_class AS predicted_label, "
    "COSINE_SIM(query_digits.pixels, reference_centroids.centroid) AS similarity, "
    "ROW_NUMBER() OVER (PARTITION BY digit_id ORDER BY similarity DESC) AS rn "
    "FROM query_digits JOIN reference_centroids "
    "ON COSINE_SIM(query_digits.pixels, reference_centroids.centroid) > 0.5"
    ") SELECT digit_id, true_label, predicted_label, similarity "
    "FROM classified WHERE rn = 1 ORDER BY digit_id"
)
sql_result = db.execute(classify_sql)
sql_rows = {row[0]: row for row in sql_result.rows}
print(f"In-process query classified {len(sql_rows)} held-out digits.")
sql_result.to_pandas().head()

In-process query classified 30 held-out digits.


,digit_id,true_label,predicted_label,similarity
0,digit_0_0,0,0,0.946915
1,digit_0_1,0,0,0.944257
2,digit_0_2,0,0,0.881193
3,digit_1_0,1,1,0.917590
4,digit_1_1,1,1,0.835758


## 5. Export the datasets directly from disk

No `/delivery` HTTP endpoint involved — `Db.dataset(name).to_pandas()`
reads `data.parquet` straight from
`{data_dir}/{db}/datasets/{name}/data.parquet` (the same on-disk layout
`/delivery` serves for the HTTP clients), since the engine and this
notebook share a filesystem.

In [5]:
query_df = db.dataset("query_digits").to_pandas()
centroids_df = db.dataset("reference_centroids").to_pandas()
print(f"query_digits: {len(query_df)} rows, reference_centroids: {len(centroids_df)} rows")
query_df.head()

query_digits: 30 rows, reference_centroids: 10 rows


,digit_id,true_label,pixels
0,digit_0_0,0,"[0.0, 1.0, 6.0, 15.0, 12.0, 1.0, 0.0, 0.0, 0.0..."
1,digit_0_1,0,"[0.0, 0.0, 10.0, 16.0, 6.0, 0.0, 0.0, 0.0, 0.0..."
2,digit_0_2,0,"[0.0, 0.0, 15.0, 2.0, 14.0, 13.0, 2.0, 0.0, 0...."
3,digit_1_0,1,"[0.0, 0.0, 0.0, 3.0, 16.0, 11.0, 1.0, 0.0, 0.0..."
4,digit_1_1,1,"[0.0, 0.0, 9.0, 13.0, 1.0, 0.0, 0.0, 0.0, 0.0,..."


## 6. Independently recompute the classification in pure numpy

Using only the raw exported vectors — not the SQL engine at all — to
confirm the in-process query's numbers are real, not an artifact of how
`COSINE_SIM`/the similarity join happen to be implemented.

In [6]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


centroid_vecs = {
    row["digit_class"]: np.array(row["centroid"], dtype=np.float64)
    for _, row in centroids_df.iterrows()
}

mismatches = []
correct = 0
for _, row in query_df.iterrows():
    digit_id = row["digit_id"]
    if digit_id not in sql_rows:
        continue
    query_vec = np.array(row["pixels"], dtype=np.float64)

    sims = {label: cosine_similarity(query_vec, c) for label, c in centroid_vecs.items()}
    best_label = max(sims, key=sims.get)
    best_sim = sims[best_label]

    _, true_label, sql_predicted, sql_similarity = sql_rows[digit_id]
    if abs(best_sim - sql_similarity) > 1e-4:
        mismatches.append(f"{digit_id}: numpy {best_sim:.6f} vs query {sql_similarity:.6f}")
    if best_label != sql_predicted:
        mismatches.append(f"{digit_id}: numpy predicted {best_label} vs query {sql_predicted}")
    if best_label == true_label:
        correct += 1

total = len(sql_rows)
sql_correct = sum(1 for row in sql_rows.values() if row[1] == row[2])
print(f"Independently-recomputed accuracy: {correct}/{total} ({100 * correct / total:.1f}%)")
print(f"In-process-query-reported accuracy: {sql_correct}/{total} ({100 * sql_correct / total:.1f}%)")

Independently-recomputed accuracy: 25/30 (83.3%)
In-process-query-reported accuracy: 25/30 (83.3%)


## 7. Verdict

In [7]:
if mismatches:
    print(f"FAIL: {len(mismatches)} mismatch(es):")
    for m in mismatches:
        print(" -", m)
    raise SystemExit(1)
elif correct != sql_correct:
    raise SystemExit("FAIL: aggregate accuracy differs between the two independently-computed paths.")
else:
    print(
        "PASS: every per-row similarity, every predicted label, and the aggregate accuracy "
        "computed from the raw exported vectors exactly match what the in-process DSL query "
        "reported -- entirely within one Python process, no server."
    )

PASS: every per-row similarity, every predicted label, and the aggregate accuracy computed from the raw exported vectors exactly match what the in-process DSL query reported -- entirely within one Python process, no server.
